# NB02 — Dimensionality Reduction: PCA and LDA

All preprocessing is demonstrated with pipelines. This notebook characterizes the representation space and saves PCA/LDA tables and figures; model selection is performed later inside nested CV.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

OUT=RESULTS/"NB02_DIMENSIONALITY"; OUT.mkdir(exist_ok=True)
FIG=FIGURES/"NB02_DIMENSIONALITY"; FIG.mkdir(exist_ok=True)
TAB=TABLES/"NB02_DIMENSIONALITY"; TAB.mkdir(exist_ok=True)

df=pd.read_excel(DATASET).rename(columns={"AspectRation":"AspectRatio"})
X=df.drop(columns=[TARGET]).copy()
y=df[TARGET].astype(str)
features=X.columns.tolist()

scaler=StandardScaler()
Xs=scaler.fit_transform(X)


In [ ]:

# PCA descriptive analysis
pca=PCA().fit(Xs)
evr=pca.explained_variance_ratio_
cum=np.cumsum(evr)
pca_var=pd.DataFrame({
    "PC":[f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio":evr,
    "cumulative_variance":cum
})
pca_var.to_csv(TAB/"pca_explained_variance.csv",index=False)

loadings=pd.DataFrame(pca.components_.T,index=features,columns=pca_var["PC"])
loadings.to_csv(TAB/"pca_loadings.csv")

scores=pca.transform(Xs)
scores_df=pd.DataFrame(scores[:,:5],columns=[f"PC{i+1}" for i in range(5)])
scores_df[TARGET]=y.values
scores_df.to_csv(OUT/"pca_scores_first5.csv",index=False)

display(pca_var)


In [ ]:

plt.figure(figsize=(7,4.5))
plt.plot(range(1,len(cum)+1),cum,marker="o")
plt.axhline(.95,linestyle="--")
plt.axhline(.99,linestyle="--")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1.02)
plt.title("PCA cumulative explained variance")
plt.tight_layout()
plt.savefig(FIG/"pca_cumulative_variance.png",dpi=300,bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,5.5))
for cls in sorted(y.unique()):
    m=(y.values==cls)
    plt.scatter(scores[m,0],scores[m,1],s=10,alpha=.45,label=cls)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA score projection")
plt.legend()
plt.tight_layout()
plt.savefig(FIG/"pca_pc1_pc2.png",dpi=300,bbox_inches="tight")
plt.show()


In [ ]:

# LDA descriptive projection (supervised; descriptive only here, not used for unbiased performance estimation)
le=LabelEncoder(); y_enc=le.fit_transform(y)
lda=LinearDiscriminantAnalysis(n_components=3)
Z=lda.fit_transform(Xs,y_enc)
lda_scores=pd.DataFrame(Z,columns=["LD1","LD2","LD3"])
lda_scores[TARGET]=y.values
lda_scores.to_csv(OUT/"lda_scores.csv",index=False)

lda_var=pd.DataFrame({
    "LD":["LD1","LD2","LD3"],
    "explained_discriminative_ratio":lda.explained_variance_ratio_,
    "cumulative":np.cumsum(lda.explained_variance_ratio_)
})
lda_var.to_csv(TAB/"lda_explained_discriminative_ratio.csv",index=False)

plt.figure(figsize=(7,5.5))
for cls in sorted(y.unique()):
    m=(y.values==cls)
    plt.scatter(Z[m,0],Z[m,1],s=10,alpha=.45,label=cls)
plt.xlabel("LD1"); plt.ylabel("LD2"); plt.title("LDA score projection")
plt.legend()
plt.tight_layout()
plt.savefig(FIG/"lda_ld1_ld2.png",dpi=300,bbox_inches="tight")
plt.show()


**Important:** the full-dataset PCA/LDA above are descriptive visualizations only. In the predictive notebooks, scaling and dimensionality reduction are fitted separately inside each training fold to prevent information leakage.